# Commute Times & Snow Emergency — Regression Discontinuity Design (RDD)
## PRACTICE SKELETON Notebook (Python + R equivalents)

**Goal:** Estimate the causal effect of a snow-emergency declaration on average commute times using a sharp Regression Discontinuity Design (RDD). The forcing variable is daily snowfall (inches); the cutpoint is 4 inches.

**How to use this notebook**
1. Read each markdown instruction carefully.
2. Fill in the `TODO` code cells.
3. Run the cell and inspect the printed output / plot.
4. When stuck, open the companion **Commute_Times_RDD_Solution.ipynb**.

**Original project language:** R.  
**This extended version:** Primary implementation in **Python**; full **R code** appears in the final section.

**Data file:** `/home/workdir/artifacts/snow.csv`


## Flowchart: Desired Outcome of the RDD Analysis

```mermaid
flowchart TD
    A[Load snow.csv] --> B[Inspect & Summary Stats]
    B --> C[EDA: Scatter + Density of Forcing Variable]
    C --> D{Continuity at cutpoint=4?}
    D -->|Yes - no manipulation| E[Choose bandwidth h]
    D -->|Suspicious| Z[Investigate / stop]
    E --> F[Visual RD plot with local linear fits]
    F --> G[Local Linear Regression<br/>Y ~ Xc + T + Xc×T | |Xc|≤h]
    G --> H[Estimate LATE τ̂ and SE]
    H --> I[Robustness: vary h, polynomial order, kernel]
    I --> J[Placebo tests at fake cutpoints]
    J --> K[Interpret causal effect of Emergency on minutes]
    K --> L[Simulation: recover known τ under different designs]
    L --> M[Report & Key Takeaways]
```

**Causal target:** Local Average Treatment Effect (LATE) of the emergency declaration for days with snowfall near 4 inches.


## 0. Setup – Libraries & Data

### TODO
1. Import `pandas as pd`, `numpy as np`, `matplotlib.pyplot as plt`, `seaborn as sns`, `statsmodels.formula.api as smf`.
2. Set a random seed and a seaborn theme.
3. Load `/home/workdir/artifacts/snow.csv` into `snow_df`.
4. Print shape, head, and column names.


In [ ]:
# TODO: imports
# import ...

# TODO: seed & theme
# np.random.seed(...)
# sns.set_theme(...)

# TODO: load data
# DATA = Path(...)
# snow_df = ...

# TODO: inspect
# print(...)


## 1. Inspect Dataframe & Summary Statistics

### TODO
1. Call `.info()` and `.describe()`.
2. Count values of the `emergency` column.
3. Group-by `emergency` and compute mean / std / count of `snowfall` and `minutes`.
4. Verify the design is **sharp**: `(snowfall >= 4) == (emergency == "Emergency")` for every row.
5. Print the min snowfall among Emergency days and the max among No-Emergency days.


In [ ]:
# TODO: info, describe, value_counts, groupby agg
# print(snow_df.info())
# ...

# TODO: sharp check
# c = 4.0
# is_sharp = ...
# print(...)


## 2. EDA – Scatter & Density of Forcing Variable

### TODO
1. Create a 1×2 figure.
2. Left panel: histogram + KDE of `snowfall`; add a vertical line at 4.
3. Right panel: scatter of `minutes` vs `snowfall` coloured / shaped by `emergency`; add the cutpoint line.
4. Add titles and show the figure.


In [ ]:
# TODO: EDA plots
# fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# sns.histplot(...)
# sns.scatterplot(...)
# plt.tight_layout()
# plt.show()
# plt.clf()


## 3. Base RD Scatter + Vertical Line at Cutpoint

### TODO
Reproduce the classic RD scatter (points coloured by emergency) and draw a dashed vertical line at the cutpoint = 4.


In [ ]:
# TODO: base scatter + cutpoint
# fig, ax = plt.subplots(figsize=(8, 5))
# sns.scatterplot(...)
# ax.axvline(...)
# plt.show()
# plt.clf()


## 4. Add Local Linear Best-Fit Lines

### TODO
1. Create columns `Xc = snowfall - 4` and `T = 1{snowfall >= 4}`.
2. Subset left side (`Xc` in [-h, 0]) and right side (`Xc` in [0, h]) with `h = 1.5`.
3. Fit `minutes ~ Xc` on each side with `smf.ols`.
4. Overlay the two fitted lines on the scatter plot.
5. Print the two intercepts (the visual jump).


In [ ]:
# TODO: local linear fits for visualisation
# h_vis = 1.5
# df = snow_df.copy()
# df["Xc"] = ...
# df["T"]  = ...
# left  = ...
# right = ...
# mL = smf.ols(...).fit()
# mR = smf.ols(...).fit()
# # plot ...
# print intercepts and jump


## 5. Bandwidth Selection & Sensitivity

### TODO
1. Loop over candidate bandwidths `[1.0, 1.5, 2.0, 2.5, 3.0]`.
2. For each `h`, restrict to `|Xc| <= h` and fit the full local-linear model `minutes ~ Xc + T + Xc:T`.
3. Print a table of h, tau, se, n, p-value.
4. Draw the scatter again and mark the window ±1.5 around the cutpoint.


In [ ]:
# TODO: bandwidth loop + visual window
# candidate_h = [1.0, 1.5, 2.0, 2.5, 3.0]
# for h in candidate_h:
#     sub = ...
#     mod = smf.ols("minutes ~ Xc + T + Xc:T", data=sub).fit()
#     print(...)
# # plot with axvline at c-h and c+h


## 6. Fit Local Linear RDD Model (main estimate)

### TODO
Using `h = 1.5`:
1. Restrict the data to the bandwidth window.
2. Fit `minutes ~ Xc + T + Xc:T`.
3. Print the full summary and extract τ̂, its SE, p-value, N and 95 % CI.


In [ ]:
# TODO: main local-linear estimate
# h = 1.5
# sub = ...
# rdd_mod = smf.ols(...).fit()
# print(rdd_mod.summary())
# print key results


## 7. Extract Key Quantities

### TODO
Print the number of observations used, the vector of standard errors, and the coefficient vector.


In [ ]:
# TODO: extract obs, se, params
# print(int(rdd_mod.nobs))
# print(rdd_mod.bse)
# print(rdd_mod.params)


## 8. Robustness Checks & Placebo Tests

### TODO
1. Build a small DataFrame of results for several bandwidths (linear specification).
2. Fit a local quadratic specification at h = 2.0 and report τ̂.
3. For placebo cutpoints 2, 3, 5, 6 (and two bandwidths), estimate the same model and print τ̂ and p-value. Expect insignificant results.


In [ ]:
# TODO: robustness + placebo
# print bandwidth sensitivity table
# quadratic model
# placebo loop over fake_c and hh


## 9. Alternate Implementation – Triangular Kernel

### TODO
Write a function `rdd_triangular(df, c=4.0, h=1.5)` that:
1. Builds Xc and T,
2. Keeps observations inside the bandwidth,
3. Assigns triangular weights `w = 1 - |Xc|/h`,
4. Fits a weighted least-squares local-linear model,
5. Returns the fitted model.

Call it and print τ̂, SE, p, N.


In [ ]:
# TODO: triangular kernel estimator
# def rdd_triangular(...):
#     ...
#     mod = smf.wls(..., weights=...).fit()
#     return mod
# mod_tri = rdd_triangular(snow_df)
# print results


## 10. Additional Practice Exercises

### TODO – Exercise A
Re-estimate the main model **without** the interaction term (common slope). Compare τ̂ to the interactive model.

### TODO – Exercise B
Restrict to the first winter (dates before 2018-11-01) and re-estimate the interactive model at h = 1.5.

### TODO – Exercise C
Create a binary outcome `long = 1{minutes > 50}` and estimate a linear-probability RD. Interpret the coefficient on T.


In [ ]:
# TODO: exercises A, B, C
# A: smf.ols("minutes ~ Xc + T", ...)
# B: filter date, re-estimate
# C: long indicator + LPM


## 11. Simulation Laboratory

### TODO
1. Write a function `simulate_rdd(n=400, c=4.0, tau_true=-10.0, noise=8.0, h=1.5, seed=None)` that:
   - Draws a forcing variable,
   - Creates a sharp treatment indicator,
   - Generates an outcome with a known jump `tau_true`,
   - Estimates the local-linear RD and returns a dict with `tau_hat`, `se`, `bias`, etc.
2. Run one recovery with the defaults and print the results.
3. Loop over a small grid of `tau_true` and `noise` values and print a recovery table.


In [ ]:
# TODO: simulation
# def simulate_rdd(...):
#     ...
#     return results_dict, sim_df
# res, sim_df = simulate_rdd(seed=123)
# print(...)
# grid experiment


## 12. Equivalent R Code (for practice in R)

Copy the R block below into an R session (or an R notebook) after installing `ggplot2`, `dplyr` and `rdd`.  
It reproduces the original Tasks 1–11 plus a few extensions.


In [ ]:
# Documentation only – does not run under the Python kernel.
# Paste the triple-quoted string into R.

r_code = '''
library(ggplot2)
library(dplyr)
library(rdd)

snow_df <- read.csv("snow.csv")
head(snow_df)

scatter_base <- ggplot(snow_df, aes(x = snowfall, y = minutes,
                                    color = emergency, shape = emergency)) +
  geom_point()
scatter_cutpoint <- scatter_base + geom_vline(xintercept = 4, linetype = "dashed")
scatter_lines <- scatter_cutpoint +
  geom_smooth(aes(group = emergency), method = "lm", se = FALSE)
print(scatter_lines)

snow_ik_bw <- IKbandwidth(X = snow_df$snowfall, Y = snow_df$minutes, cutpoint = 4)
print(snow_ik_bw)

snow_rdd <- RDestimate(formula = minutes ~ snowfall,
                       cutpoint = 4, bw = snow_ik_bw, data = snow_df)
print(snow_rdd)
print(snow_rdd$obs)
print(snow_rdd$se)

# Placebo
print(RDestimate(minutes ~ snowfall, cutpoint = 2, data = snow_df))
'''
print("R code ready for copy-paste (see variable r_code or the source).")


## Next Steps

- Compare your answers with **Commute_Times_RDD_Solution.ipynb**.
- Experiment in the Simulation section: what happens when you set `noise=20` or `n=100`?
- Think about external validity: would the same LATE apply on a day with 10 inches of snow?
